In [ ]:
# ── KAGGLE SETUP (only runs on Kaggle, no-op everywhere else) ────────────────
import os, sys
from pathlib import Path

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/input").exists():
    import subprocess

    REPO_URL = "https://github.com/<your-username>/fake-news-detection.git"
    CLONE_DIR = Path("/kaggle/working/fake-news-detection")

    if not CLONE_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(CLONE_DIR)], check=True)
        print(f"✅ Cloned repo → {CLONE_DIR}")
    else:
        print(f"✅ Repo already at {CLONE_DIR}")

    _KAGGLE_SKIP = {"numpy", "torch", "torchvision", "jupyter", "jupyterlab", "ipykernel"}
    def _is_skipped(line):
        s = line.strip()
        if not s or s.startswith("#"):
            return True
        pkg = s.split(">=")[0].split("==")[0].split("<")[0].split(">")[0].split("[")[0].strip().lower()
        return pkg in _KAGGLE_SKIP

    req_lines = (CLONE_DIR / "requirements.txt").read_text().splitlines()
    filtered = [l for l in req_lines if not _is_skipped(l)]
    filtered_req = Path("/kaggle/working/_requirements_filtered.txt")
    filtered_req.write_text("\n".join(filtered))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(filtered_req)], check=True)
    print(f"✅ requirements installed (skipped: {sorted(_KAGGLE_SKIP)})")

    def _find_data_root():
        for d in Path("/kaggle/input").rglob("data/json"):
            if d.is_dir():
                return d.parent.parent
        return None

    data_root = _find_data_root()
    if data_root is None:
        raise RuntimeError("Could not find data/json/ under /kaggle/input/")
    print(f"✅ Data found at: {data_root}")

    env_content = f"DATA_ROOT={data_root}\nPLATFORM=kaggle\n"
    (CLONE_DIR / ".env.kaggle").write_text(env_content)
    print(f"✅ Checkpoints → /kaggle/working/checkpoints_coolant_end2end/")
else:
    print("Not Kaggle — setup skipped.")

In [ ]:
# ─── Environment Setup (do not edit) ────────────────────────────────────────
import os, sys
from pathlib import Path

def _detect_platform():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/input").exists():
        return "kaggle", False
    try:
        import google.colab
        return "colab", False
    except ImportError:
        pass
    if Path("/workspace").exists() and os.environ.get("VAST_CONTAINERLABEL"):
        return "vastai", False
    if Path("/workspace").exists():
        return "vastai", True
    if sys.platform == "win32":
        return "windows", False
    if sys.platform == "darwin":
        return "mac", False
    return None, True

PLATFORM, _uncertain = _detect_platform()

if PLATFORM == "colab":
    from google.colab import drive
    drive.mount("/content/drive")

try:
    _nb_path = Path(__file__).resolve()
except NameError:
    _nb_path = Path.cwd()

if PLATFORM == "colab":
    PROJECT_ROOT = Path("/content/drive/MyDrive/Thesis_Final/fake-news-detection")
elif PLATFORM == "kaggle":
    PROJECT_ROOT = Path(os.environ.get("KAGGLE_PROJECT_ROOT", "/kaggle/working/fake-news-detection"))
else:
    PROJECT_ROOT = _nb_path.parents[1]

sys.path.insert(0, str(PROJECT_ROOT))

_env_map = {
    "colab": PROJECT_ROOT / ".env.colab",
    "kaggle": PROJECT_ROOT / ".env.kaggle",
    "vastai": PROJECT_ROOT / ".env.vastai",
    "windows": PROJECT_ROOT / ".env.windows",
    "mac": PROJECT_ROOT / ".env.mac",
}

if PLATFORM is None:
    _env_file = PROJECT_ROOT / ".env"
elif _uncertain:
    _env_file = _env_map["vastai"]
else:
    _env_file = _env_map[PLATFORM]

from dotenv import load_dotenv
if not _env_file.exists():
    _fallback = PROJECT_ROOT / ".env"
    if _fallback.exists():
        _env_file = _fallback
    else:
        raise FileNotFoundError(f"No .env file found: {_env_file}")
load_dotenv(_env_file, override=True)

from src.utils.env_utils import get_data_root
DATA_ROOT = get_data_root()

print(f"✅ Platform : {PLATFORM or 'unknown'}")
print(f"✅ DATA_ROOT: {DATA_ROOT}")

# COOLANT End-to-End Training — Stage 1 Multimodal Alignment

**Khác biệt chính với `03_coolant_training.ipynb`**:

| | Baseline | End-to-End (notebook này) |
|---|---|---|
| **Input** | Pre-extracted features (HDF5) | Raw images + raw text |
| **Feature extractors** | Frozen (PhoBERT, ResNet50) | **Trainable** — fine-tune cùng COOLANT |
| **Bottleneck** | Có — feature đã cố định | **Không** — gradient chảy ngược về raw data |
| **Batch size** | 32 | 4 (với grad accum 4 → eff 16) |
| **Mixed precision** | Không | **AMP (FP16)** — tiết kiệm VRAM |
| **GPU yêu cầu** | ~2GB | ~12GB (T4/P100) |

**Pipeline**: Raw image/text → PhoBERT + ResNet50 (trainable) → COOLANT → prediction

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────────
import os
from pathlib import Path

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/input").exists():
    PROJECT_ROOT = Path(os.environ.get("KAGGLE_PROJECT_ROOT", "/kaggle/working/fake-news-detection"))
else:
    PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == "pipeline" else Path.cwd()

try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env", override=False)
except ImportError:
    pass

DATA_ROOT = Path(os.environ["DATA_ROOT"]) if os.environ.get("DATA_ROOT") else PROJECT_ROOT

CONFIG = {
    "paths": {
        "json_dir": DATA_ROOT / "data" / "json",
        "jpg_dir": DATA_ROOT / "data" / "jpg",
        "checkpoint_root": DATA_ROOT / "training" / "checkpoints_coolant_end2end",
        "mlflow_dir": DATA_ROOT / "mlruns",
    },
    "model": {
        "phobert_name": "vinai/phobert-base-v2",
        "max_text_len": 256,
        "image_size": 224,
        # COOLANT architecture (same as baseline)
        "shared_dim": 128,
        "sim_dim": 64,
        "clip_embed_dim": 64,
        "feature_dim": 96,
        "h_dim": 64,
        # Optimizer
        "lr": 5e-5,  # lower LR for fine-tuning pretrained encoders
        "lr_backbone": 1e-5,  # even lower for PhoBERT/ResNet50
        "weight_decay": 1e-5,
        "dropout": 0.1,
    },
    "training": {
        "batch_size": 4,
        "grad_accumulation_steps": 4,  # effective batch = 16
        "max_epochs": 10,
        "patience": 5,
        "negative_shift": 1,  # smaller shift for smaller dataset
        "min_batch_for_negatives": 2,
        "grad_clip": 1.0,
        "warmup_epochs": 1,
        "seed": 42,
        "use_amp": True,  # FP16 mixed precision
        # Loss weights
        "similarity_weight": 0.5,
        "clip_weight": 0.2,
        "detection_weight": 1.0,
        # EMA
        "use_ema": True,
        "ema_decay": 0.999,
    },
    "loss": {
        "cosine_margin": 0.2,
        "label_smoothing": 0.0,
    },
    "mlflow": {"experiment_name": "coolant-end2end"},
    "checkpointing": {
        "selection_metric": "val_accuracy",
        "checkpoint_every": 3,
    },
    "safety": {
        "smoke_test": False,
        "smoke_batches": 2,
        "auto_install_deps": False,
        "resume_from_checkpoint": None,
    },
}

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/input").exists():
    CONFIG["paths"]["checkpoint_root"] = Path("/kaggle/working/checkpoints_coolant_end2end")
    CONFIG["paths"]["mlflow_dir"] = Path("/kaggle/working/mlruns")

In [ ]:
# ── IMPORTS ────────────────────────────────────────────────────────────────
import sys, os, gc, json, random, hashlib
from datetime import datetime
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from PIL import Image
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

# HuggingFace
from transformers import AutoTokenizer, AutoModel
import torchvision.transforms as T
import torchvision.models as models

# AdaBelief
try:
    from adabelief_pytorch import AdaBelief
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "adabelief-pytorch"])
    from adabelief_pytorch import AdaBelief

_root = PROJECT_ROOT
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

CONFIG["paths"]["checkpoint_root"].mkdir(parents=True, exist_ok=True)

from src.models.coolant_official import COOLANT_Official, GatedMLP
from src.preprocessing.coolant.training_utils import (
    make_coolant_pairs,
    make_detection_batch,
    soft_cross_entropy,
)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Step 1: Raw Dataset — JSON → PIL Images + Text Strings

In [ ]:
class EndToEndDataset(Dataset):
    """Load raw JSON articles, return PIL images + text strings + labels."""

    def __init__(self, json_path: Path, jpg_dir: Path, split: str = "train"):
        with open(json_path, "r", encoding="utf-8") as f:
            articles = json.load(f)

        self.samples = []
        for article in articles:
            title = article.get("title", "")
            content = article.get("content", "") or article.get("text", "")
            text = f"{title}. {content}".strip() if title else content
            if not text.strip():
                continue

            images = article.get("images", [])
            img_path = None
            if images and len(images) > 0:
                first = images[0]
                if isinstance(first, dict):
                    img_path = first.get("folder_path", "")
                elif isinstance(first, str):
                    img_path = first

            label = article.get("label", 0)
            if isinstance(label, str):
                label = 1 if label.lower() in ["fake", "false", "1"] else 0

            self.samples.append({
                "text": text.strip(),
                "img_path": img_path,
                "label": int(label),
            })

        self.jpg_dir = jpg_dir
        self.split = split
        print(f"EndToEndDataset [{split}]: {len(self.samples)} samples")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        text = s["text"]

        # Load image — fallback to black image if missing
        img_path = s["img_path"]
        if img_path:
            full_path = self.jpg_dir / img_path
            if full_path.exists():
                try:
                    img = Image.open(full_path).convert("RGB")
                except Exception:
                    img = Image.new("RGB", (224, 224), (0, 0, 0))
            else:
                img = Image.new("RGB", (224, 224), (0, 0, 0))
        else:
            img = Image.new("RGB", (224, 224), (0, 0, 0))

        return text, img, s["label"]


# ── Build datasets ─────────────────────────────────────────────────────────
json_dir = CONFIG["paths"]["json_dir"]
jpg_dir = CONFIG["paths"]["jpg_dir"]

datasets = {}
for split in ["train", "dev", "test"]:
    json_path = json_dir / f"news_data_vifactcheck_{split}_labeled.json"
    if not json_path.exists():
        json_path = json_dir / f"news_data_vifactcheck_{split}_cleaned.json"
    if json_path.exists():
        datasets[split] = EndToEndDataset(json_path, jpg_dir, split)

print(f"\nSplits loaded: {list(datasets.keys())}")

## Step 2: End-to-End Model — PhoBERT + ResNet50 + COOLANT

In [ ]:
class EndToEndCOOLANT(nn.Module):
    """
    End-to-end COOLANT with trainable feature extractors.

    Raw image/text → PhoBERT + ResNet50 (trainable) → COOLANT → prediction
    """

    def __init__(self, config):
        super().__init__()

        # ── Text encoder: PhoBERT-base ────────────────────────────────────
        phobert_name = config["model"]["phobert_name"]
        self.tokenizer = AutoTokenizer.from_pretrained(phobert_name)
        self.text_encoder = AutoModel.from_pretrained(phobert_name)
        self.max_text_len = config["model"]["max_text_len"]

        # ── Image encoder: ResNet50 (no final FC) ─────────────────────────
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.image_encoder = nn.Sequential(*list(backbone.children())[:-1])  # → [B, 2048, 1, 1]

        # Image preprocessing
        self.image_transform = T.Compose([
            T.Resize((config["model"]["image_size"], config["model"]["image_size"])),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        # ── COOLANT core (same as baseline) ───────────────────────────────
        self.coolant = self._build_coolant(config)

        # Loss weights
        self.similarity_weight = config["training"]["similarity_weight"]
        self.clip_weight = config["training"]["clip_weight"]

    def _build_coolant(self, config):
        """Build COOLANT core without its own EncodingPart (we replace it)."""
        from src.models.coolant_official import (
            SimilarityModule, CLIP, DetectionModule,
            EncodingPart, UnimodalDetection, CrossModule4Batch,
            AmbiguityLearning,
        )
        from src.models.senet import SEAttentionModule

        mc = config["model"]
        # Use 768 (PhoBERT hidden) as text_input_dim for COOLANT
        text_dim = 768
        img_dim = 2048

        self.similarity_module = SimilarityModule(
            shared_dim=mc["shared_dim"], sim_dim=mc["sim_dim"],
            text_input_dim=text_dim, image_input_dim=img_dim,
        )
        self.clip_module = CLIP(
            embed_dim=mc["clip_embed_dim"],
            text_input_dim=text_dim, image_input_dim=img_dim,
        )
        self.detection_module = DetectionModule(
            feature_dim=mc["feature_dim"], h_dim=mc["h_dim"],
            text_input_dim=text_dim, image_input_dim=img_dim,
        )

    def encode_text(self, text_batch):
        """Tokenize and encode raw text → [B, 768, seq_len]."""
        tokens = self.tokenizer(
            text_batch, padding=True, truncation=True,
            max_length=self.max_text_len, return_tensors="pt",
        )
        device = next(self.text_encoder.parameters()).device
        tokens = {k: v.to(device) for k, v in tokens.items()}
        with torch.no_grad() if not self.training else torch.enable_grad():
            out = self.text_encoder(**tokens)
        # [B, seq_len, 768] → transpose to [B, 768, seq_len] (COOLANT expects this)
        return out.last_hidden_state.transpose(1, 2)

    def encode_image(self, image_batch):
        """Preprocess and encode raw PIL images → [B, 2048]."""
        device = next(self.image_encoder.parameters()).device
        tensors = torch.stack([self.image_transform(img) for img in image_batch]).to(device)
        features = self.image_encoder(tensors)  # [B, 2048, 1, 1]
        return features.flatten(1)  # [B, 2048]

    def forward(self, text_batch, image_batch, return_all=False):
        text_feat = self.encode_text(text_batch)
        image_feat = self.encode_image(image_batch)
        return self.coolant_forward(text_feat, image_feat, return_all)

    def coolant_forward(self, text_raw, image_raw, return_all=False):
        """COOLANT core forward (same as COOLANT_Official.forward)."""
        text_aligned_sim, image_aligned_sim, similarity_pred = self.similarity_module(text_raw, image_raw)
        image_aligned_clip, text_aligned_clip = self.clip_module(image_raw, text_raw)
        detection_logits, attention_weights, ambiguity_weights = self.detection_module(
            text_raw, image_raw, text_aligned_clip, image_aligned_clip
        )
        outputs = {
            "similarity_pred": similarity_pred,
            "detection_logits": detection_logits,
            "attention_weights": attention_weights,
            "ambiguity_weights": ambiguity_weights,
            "text_aligned_clip": text_aligned_clip,
            "image_aligned_clip": image_aligned_clip,
        }
        if return_all:
            outputs.update({
                "text_aligned_sim": text_aligned_sim,
                "image_aligned_sim": image_aligned_sim,
                "text_raw": text_raw,
                "image_raw": image_raw,
            })
        return outputs

    def compute_clip_loss(self, text_features, image_features):
        logits = torch.matmul(image_features, text_features.T) * torch.exp(self.clip_module.temperature)
        batch_size = text_features.size(0)
        labels = torch.arange(batch_size, device=text_features.device)
        return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2

    def compute_detection_loss(self, logits, labels, attention_w, ambiguity_w):
        ce = F.cross_entropy(logits, labels)
        kl = F.kl_div(
            F.log_softmax(logits, dim=1),
            F.softmax(logits.detach(), dim=1),
            reduction="batchmean",
        )
        return ce + 0.1 * kl

    def get_param_groups(self, lr, lr_backbone, wd):
        """Separate param groups: backbone (lower LR) vs COOLANT head (higher LR)."""
        backbone_params = list(self.text_encoder.parameters()) + list(self.image_encoder.parameters())
        head_params = [p for n, p in self.named_parameters()
                       if not n.startswith(("text_encoder.", "image_encoder."))]
        return [
            {"params": backbone_params, "lr": lr_backbone, "weight_decay": wd},
            {"params": head_params, "lr": lr, "weight_decay": wd},
        ]


print("Building model...")
model = EndToEndCOOLANT(CONFIG)
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total/1e6:.1f}M  Trainable: {trainable/1e6:.1f}M")

## Step 3: Device, Seed, DataLoaders

In [ ]:
def select_device():
    if torch.cuda.is_available():
        dev = torch.device("cuda")
        mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"Device: cuda ({torch.cuda.get_device_name(0)}, {mem:.1f} GB)")
    else:
        dev = torch.device("cpu")
        print("Device: cpu — End-to-end training requires GPU!")
    return dev

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

DEVICE = select_device()
seed_everything(CONFIG["training"]["seed"])

def collate_fn(batch):
    texts, images, labels = zip(*batch)
    return list(texts), list(images), torch.tensor(labels)

loaders = {}
for split, ds in datasets.items():
    shuffle = (split == "train")
    bs = CONFIG["training"]["batch_size"] if split == "train" else CONFIG["training"]["batch_size"] * 2
    loaders[split] = DataLoader(ds, batch_size=bs, shuffle=shuffle, collate_fn=collate_fn, num_workers=2)
    print(f"  {split}: {len(loaders[split])} batches (bs={bs})")

model = model.to(DEVICE)
print(f"Model on {DEVICE}")

## Step 4: Losses, Optimizer, Scheduler

In [ ]:
# Losses
loss_cos = nn.CosineEmbeddingLoss(margin=CONFIG["loss"]["cosine_margin"])

# AdaBelief with differential LR
param_groups = model.get_param_groups(
    CONFIG["model"]["lr"],
    CONFIG["model"]["lr_backbone"],
    CONFIG["model"]["weight_decay"],
)
optimizer = AdaBelief(
    param_groups,
    eps=1e-16,
    betas=(0.9, 0.999),
    weight_decouple=True,
    rectify=True,
    print_change_log=False,
)

def make_warmup_cosine_scheduler(optimizer, warmup_epochs, total_epochs):
    def _lr_lambda(current_epoch):
        if current_epoch < warmup_epochs:
            return float(current_epoch + 1) / max(1, warmup_epochs)
        progress = float(current_epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return max(0.0, 0.5 * (1.0 + np.cos(np.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, _lr_lambda)

scheduler = make_warmup_cosine_scheduler(
    optimizer, CONFIG["training"]["warmup_epochs"], CONFIG["training"]["max_epochs"]
)

# AMP scaler
scaler = GradScaler() if CONFIG["training"]["use_amp"] and DEVICE.type == "cuda" else None

print(f"Optimizer: AdaBelief (head_lr={CONFIG['model']['lr']}, backbone_lr={CONFIG['model']['lr_backbone']})")
print(f"AMP: {'enabled' if scaler else 'disabled'}")

## Step 5: EMA, Training & Evaluation Functions

In [ ]:
class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        self._register()

    def _register(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone().detach()

    def update(self):
        with torch.no_grad():
            for name, param in self.model.named_parameters():
                if param.requires_grad:
                    self.shadow[name].mul_(self.decay).add_(param.data, alpha=1.0 - self.decay)

    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data.copy_(self.shadow[name])

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data.copy_(self.backup[name])
        self.backup.clear()


def compute_metrics(y_true, y_pred, prefix):
    acc = (np.array(y_pred) == np.array(y_true)).mean()
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    return {
        f"{prefix}_accuracy": round(float(acc), 4),
        f"{prefix}_macro_f1": round(float(macro_f1), 4),
    }


def train_one_epoch(epoch, model, loaders, optimizer, scheduler, device, config, ema=None):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []
    n_batches = 0
    accum_steps = config["training"]["grad_accumulation_steps"]
    grad_clip = config["training"]["grad_clip"]
    use_amp = scaler is not None

    pbar = tqdm(loaders["train"], desc=f"Epoch {epoch:02d} [train]", leave=False)
    for batch_idx, (texts, images, labels) in enumerate(pbar):
        labels = labels.to(device)

        # ── Forward through end-to-end model ─────────────────────────────
        if use_amp:
            with autocast():
                out = model(texts, images, return_all=True)
                loss = _compute_losses(model, out, labels, device, config)
                loss = loss / accum_steps
            scaler.scale(loss).backward()
        else:
            out = model(texts, images, return_all=True)
            loss = _compute_losses(model, out, labels, device, config)
            loss = (loss / accum_steps).backward()

        # ── Optimizer step ───────────────────────────────────────────────
        if (batch_idx + 1) % accum_steps == 0:
            if use_amp:
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            if use_amp:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad()
            if ema is not None:
                ema.update()

        total_loss += loss.item() * accum_steps
        n_batches += 1
        preds = out["detection_logits"].argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        pbar.set_postfix(loss=f"{loss.item() * accum_steps:.3f}")

    metrics = compute_metrics(all_labels, all_preds, "train")
    metrics["train_loss"] = round(total_loss / max(1, n_batches), 4)
    return metrics


def _compute_losses(model, out, labels, device, config):
    """Compute paper-faithful COOLANT losses."""
    t = config["training"]

    # L_Con: CLIP contrastive on CLIP module outputs
    ta_m, ia_m = out["text_aligned_clip"], out["image_aligned_clip"]
    lbl_m = torch.ones(ta_m.size(0), device=device)
    sim_loss = loss_cos(ta_m, ia_m, lbl_m)
    clip_loss = model.compute_clip_loss(ta_m, ia_m)

    # L_SM: Cross-module soft distillation (SimilarityModule → CLIP module, bidirectional)
    if t.get("soft_clip_distillation", True):
        temp = torch.exp(model.clip_module.temperature)
        sim_ta = out["text_aligned_sim"].detach()
        sim_ia = out["image_aligned_sim"].detach()
        sim_logits = torch.matmul(sim_ia, sim_ta.T) * temp
        soft_tgt = F.softmax(sim_logits, dim=1)
        clip_logits = torch.matmul(ia_m, ta_m.T) * temp
        soft_loss = (soft_cross_entropy(clip_logits, soft_tgt) + soft_cross_entropy(clip_logits.T, soft_tgt.T)) * 0.5
    else:
        soft_loss = 0.0

    # Detection loss
    det_loss = model.compute_detection_loss(
        out["detection_logits"], labels, out["attention_weights"], out["ambiguity_weights"]
    )

    return t["similarity_weight"] * sim_loss + t["clip_weight"] * (clip_loss + soft_loss) + t["detection_weight"] * det_loss


@torch.no_grad()
def evaluate(model, loader, device, config, split_name):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    n_batches = 0

    for texts, images, labels in tqdm(loader, desc=f"  [{split_name}]", leave=False):
        labels = labels.to(device)
        out = model(texts, images, return_all=False)
        loss = model.compute_detection_loss(
            out["detection_logits"], labels, out["attention_weights"], out["ambiguity_weights"]
        )
        total_loss += loss.item()
        n_batches += 1
        all_preds.extend(out["detection_logits"].argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    metrics = compute_metrics(all_labels, all_preds, split_name)
    metrics[f"{split_name}_loss"] = round(total_loss / max(1, n_batches), 4)
    return metrics

print("Functions defined.")

## Step 6: MLflow & Run Setup

In [ ]:
import mlflow

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"coolant_end2end_{timestamp}"
run_dir = CONFIG["paths"]["checkpoint_root"] / run_name
artifact_dir = run_dir / "artifacts"
run_dir.mkdir(parents=True, exist_ok=True)
artifact_dir.mkdir(parents=True, exist_ok=True)
print(f"Run dir: {run_dir}")

mlflow_enabled = False
try:
    mlflow.set_tracking_uri(CONFIG["paths"]["mlflow_dir"].as_uri())
    mlflow.set_experiment(CONFIG["mlflow"]["experiment_name"])
    mlflow.start_run(run_name=run_name)
    mlflow.log_params({k: str(v) for k, v in CONFIG["training"].items()})
    mlflow_enabled = True
except Exception:
    pass

## Step 7: Run Training

In [ ]:
# ── EMA setup ───────────────────────────────────────────────────────────────
ema = None
if CONFIG["training"].get("use_ema", False):
    ema = ModelEMA(model, decay=CONFIG["training"]["ema_decay"])
    print(f"EMA enabled (decay={CONFIG['training']['ema_decay']})")

history = []
best_val_acc = -1.0
best_epoch = -1
best_ckpt_path = run_dir / "best_model.pth"
patience_counter = 0

print(f"Starting training — max_epochs={CONFIG['training']['max_epochs']}, device={DEVICE}")
print()

try:
    for epoch in range(CONFIG["training"]["max_epochs"]):
        train_metrics = train_one_epoch(epoch, model, loaders, optimizer, scheduler, DEVICE, CONFIG, ema=ema)
        val_metrics = evaluate(model, loaders["dev"], DEVICE, CONFIG, "val")
        scheduler.step()

        epoch_record = {
            "epoch": epoch,
            "train_loss": train_metrics["train_loss"],
            "train_accuracy": train_metrics["train_accuracy"],
            "val_loss": val_metrics["val_loss"],
            "val_accuracy": val_metrics["val_accuracy"],
            "val_macro_f1": val_metrics["val_macro_f1"],
            "lr": optimizer.param_groups[0]["lr"],
        }

        val_acc = val_metrics["val_accuracy"]
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            patience_counter = 0
            if ema is not None:
                ema.apply_shadow()
            torch.save({"model_state_dict": model.state_dict(), "epoch": epoch, "config": CONFIG}, best_ckpt_path)
            if ema is not None:
                ema.restore()
            print(f"  ★ Best val_accuracy={val_acc:.4f} at epoch {epoch} → best_model.pth")
        else:
            patience_counter += 1

        history.append(epoch_record)
        print(f"Epoch {epoch:02d} | train_loss={train_metrics['train_loss']:.4f} train_acc={train_metrics['train_accuracy']:.4f} val_loss={val_metrics['val_loss']:.4f} val_acc={val_acc:.4f} (patience {patience_counter}/{CONFIG['training']['patience']})")

        if patience_counter >= CONFIG["training"]["patience"]:
            print(f"Early stopping at epoch {epoch}")
            break

except KeyboardInterrupt:
    print("Interrupted.")

print(f"\nTraining complete. Best val_accuracy={best_val_acc:.4f} at epoch {best_epoch}.")

## Step 8: Test Evaluation & Export

In [ ]:
print(f"Loading best checkpoint: {best_ckpt_path}")
ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])

print("\nRunning test evaluation...")
test_metrics = evaluate(model, loaders["test"], DEVICE, CONFIG, "test")
for k, v in test_metrics.items():
    print(f"  {k}: {v}")

# ── Export ─────────────────────────────────────────────────────────────────
import shutil
run_name = run_dir.name

if PLATFORM == "kaggle":
    zip_path = Path("/kaggle/working") / f"{run_name}.zip"
    shutil.make_archive(str(zip_path.with_suffix("")), "zip", run_dir)
    print(f"\nCheckpoint zipped: {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")
    print("Download from Kaggle notebook output panel.")
elif PLATFORM == "colab":
    drive_dir = Path("/content/drive/MyDrive/Thesis_Final/training/checkpoints_coolant_end2end") / run_name
    drive_dir.parent.mkdir(parents=True, exist_ok=True)
    if drive_dir.exists():
        shutil.rmtree(drive_dir)
    shutil.copytree(run_dir, drive_dir)
    print(f"Copied to Google Drive: {drive_dir}")
else:
    print(f"Checkpoint at: {run_dir}")